In [112]:
# Install required libraries if not already installed
%pip install -q langchain langchain-community langchain-classic faiss-cpu pypdf wikipedia python-docx sentence-transformers


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Agentic AI Chatbot with Tool Use

This notebook demonstrates an agentic AI chatbot using LM Studio, LangChain, and various tools for PDF analysis, Wikipedia search, arithmetic reasoning, and documentation creation.

In [113]:
# Imports and Setup
import os
import wikipedia
import textwrap
from langchain_classic.memory import ConversationBufferMemory
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from docx import Document

# LM Studio API (assume local server, e.g., http://localhost:1234)
import requests

# Memory for conversation
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Emmbeddings for vector storex
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# PDF Vector Store (initialized later if PDF found)
def initialize_pdf_system():
    """Finds a PDF in the current folder and prepares the vector store."""
    # Use os.getcwd() instead of __file__ for Notebook compatibility
    try:
        current_dir = os.getcwd() 
        pdf_files = [f for f in os.listdir(current_dir) if f.lower().endswith('.pdf')]
        
        if not pdf_files:
            return None

        target_pdf = os.path.join(current_dir, pdf_files[0])
        
        # --- The rest of your LangChain logic stays the same ---
        loader = PyPDFLoader(target_pdf)
        raw_docs = loader.load()
        
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
        docs = splitter.split_documents(raw_docs)
        
        return FAISS.from_documents(docs, embeddings)
        
    except Exception as e:
        print(f"[Error] Failed to index PDF: {e}")
        return None

# Print helper for long text
def print_wrapped(text, width=100):
    wrapper = textwrap.TextWrapper(width=width, replace_whitespace=False)
    lines = text.split('\n')
    wrapped_text = [wrapper.fill(line) for line in lines]
    print("\n".join(wrapped_text))

# Helper for docx creation
def create_docx(text, filename="output.docx"):
    doc = Document()
    doc.add_paragraph(text)
    doc.save(filename)
    return filename

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6167.74it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [114]:
# LM Studio Chatbot interaction

def lm_studio_chat(prompt, history=None, endpoint="http://localhost:1234/v1/chat/completions"):
    messages = []

    if history:
        for h in history:
            messages.append({"role": "user", "content": h[0]})
            messages.append({"role": "assistant", "content": h[1]})

    # always include current prompt
    messages.append({"role": "user", "content": prompt})

    payload = {
        "messages": messages,
        "model": "local-model",
        "stream": False
    }

    response = requests.post(endpoint, json=payload)
    return response.json()["choices"][0]["message"]["content"]

print("LM Studio chat function ready.")

LM Studio chat function ready.


In [115]:
# Wiki search helper

def wiki_search(query):
    try:
        page = wikipedia.page(query, auto_suggest=True)
        return page.content  # full article text
    except wikipedia.exceptions.DisambiguationError as e:
        return f"Multiple results found: {e.options[:5]}"
    except wikipedia.exceptions.PageError:
        return "No page found for that query."
    except Exception as e:
        return f"Wiki search error: {e}"
    
print("Wiki search function ready.")

Wiki search function ready.


In [ ]:
# Main chatbot loop

import re

conversation_stats = {
    "user_turns": 0,
    "wiki_searches": 0,
    "pdf_queries": 0,
    "arithmetic": 0,
    "ai_chats": 0
}
conversation_history = []

while True:
    q = input("You: [q to quit] ")
    print("\n\n[User] \n" + q)
    if q.strip().lower() == "q":
        print("\n--- Conversation Statistics ---")
        for k, v in conversation_stats.items():
            print(f"{k}: {v}")
        print("Goodbye!")
        break
    conversation_stats["user_turns"] += 1
    
    # LM Studio decides tool
    lm_response = lm_studio_chat(f"Analyze the user input and decide the tool to use: chat, wiki, arithmetic, pdf. Input: {q}. Output format: TOOL: <tool> | KEYWORDS: <keywords>")
    print(f"[LM Studio Tool Decision] {lm_response}")
    tool_match = re.search(r'TOOL: (\w+)', lm_response)
    tool = tool_match.group(1).lower() if tool_match else "chat"
    
    # Wiki search is a simple retrieval based on keywords extracted from LM response
    if tool == "wiki":
        conversation_stats["wiki_searches"] += 1
        keywords_match = re.search(r'KEYWORDS: (.+)', lm_response)
        keywords = keywords_match.group(1) if keywords_match else q
        wiki_result = wiki_search(keywords)
        print("[Wiki]")
        print_wrapped(wiki_result)
        doc_choice = input("Create documentation in docx? (y/n): ")
        if doc_choice.lower().startswith("y"):
            lm_response_doc = lm_studio_chat(f"Format to documentation in docx style. Input: {wiki_result}")
            create_docx(lm_response_doc)
            print("\nDocx created as output.docx")
    
    # PDF analyzer is a vector search over the PDF content, then LM Studio analyzes the retrieved content to answer the question
    elif tool == "pdf":
        conversation_stats["pdf_queries"] += 1
        
        if pdf_vectordb is None:
            print("[PDF Analyzer] Searching folder and indexing PDF...")
            pdf_vectordb = initialize_pdf_system()

        if pdf_vectordb is None:
            print("[PDF Analyzer] No PDF file found in folder or indexing failed.\n")
        else:
            search_results = pdf_vectordb.similarity_search(q, k=3)
            
            context = "\n\n".join([doc.page_content for doc in search_results])
            
            prompt = (
                f"You are a helpful assistant analyzing a PDF document.\n"
                f"Use the following context to answer the question.\n\n"
                f"Context:\n{context}\n\n"
                f"Question: {q}\n\n"
                f"Answer based strictly on the context provided above:"
            )
            
            # 6. Send to LM Studio and display
            lm_response_pdf = lm_studio_chat(prompt)
            
            print("[PDF Analyzer]")
            print_wrapped(lm_response_pdf)

    # Arithmetic is a direct eval (not recommended for production, but simple for this example)
    elif tool == "arithmetic":
        conversation_stats["arithmetic"] += 1
        
        lm_response_arith = lm_studio_chat(f"Evaluate the following arithmetic expression, reason about it and correctly answer: {q}")
        print(f"[Arithmetic] ")
        print_wrapped(lm_response_arith)
    
    # Default to LM Studio chat response
    else:
        conversation_stats["ai_chats"] += 1
        answer = lm_studio_chat(q, history=conversation_history)
        print(f"[AI] ")
        print_wrapped(answer)
    
    # Update conversation history
    conversation_history.append((q, answer))



[User] 
hello my name is marc
[LM Studio Tool Decision] TOOL: chat | KEYWORDS: greeting, self-introduction, conversation
[AI] 
Hello Marc! It's nice to meet you. How can I help you today?


[User] 
calculate the circumference of 4 circle conjoined on each axis
[LM Studio Tool Decision] TOOL: arithmetic | KEYWORDS: calculate, circumference, circle, 4 circles
[Arithmetic] 
This prompt presents a geometric problem disguised as an arithmetic expression. To provide a correct
evaluation, I must first identify crucial missing information and then establish assumptions
regarding the intended scope of the geometry.

**Conclusion: A numerical answer is impossible without knowing the radius or diameter ($d$) of the
circles.**

However, I can thoroughly reason through the problem and provide the exact formula needed once that
variable is provided.

***

### ⚙️ Reasoning and Analysis

#### 1. Identify Geometric Variables
*   **Object:** Circles.
*   **Measure Required:** Circumference ($C$).
*   